[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/cours/seance3_cours.ipynb)

# Séance 4.3 — Arbres de décision — comprendre les variables qui déterminent la prédiction

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- lire un arbre de décision comme une suite de règles métier
- choisir la profondeur d'un arbre par validation croisée, sans toucher au test
- dire pourquoi l'importance native d'un arbre est biaisée
- mesurer l'importance d'une variable par permutation, sur le jeu de test
- appliquer cette mesure à n'importe quel modèle, arbre ou régression
- montrer le **sens** d'un effet avec une dépendance partielle

## La question que le comité pose toujours

En séance 4.2, le modèle a désigné 1 073 abonnés à rappeler. En comité, la
question suivante tombe immanquablement :

> *« D'accord. Mais **qu'est-ce qui** fait qu'un client part ? »*

Un modèle qui prédit sans expliquer ne se déploie pas : personne n'engage un
budget sur une boîte noire. Cette séance répond à la question — et montre au
passage que **la première réponse qu'on obtient est fausse**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import accuracy_score, f1_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
tel = pd.read_csv(BASE + "churn.csv")
tel["total"] = pd.to_numeric(tel["total"], errors="coerce")
tel = tel.dropna(subset=["total"])

y = tel["churn"]
# .astype(float) : les dependances partielles refusent les colonnes
# entieres, et les 0/1 de get_dummies sont des booleens
X = pd.get_dummies(tel.drop(columns=["churn"]), drop_first=True).astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print(X.shape[1], "variables")

## 1. Un arbre se lit

Un arbre de décision pose des questions en cascade — c'est le modèle que la
séance 4.1 dessinait en escalier. Ici, il ne prédit plus un montant mais une
**décision** : cet abonné part-il, oui ou non ?

Sa force n'est pas sa performance : c'est qu'on peut le **lire**.

In [ ]:
arbre = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)

# export_text : l'arbre en toutes lettres, une regle par ligne
print(export_text(arbre, feature_names=list(X.columns)))

Lisez la première ligne à voix haute : *« le client est-il au contrat
mensuel ? »* L'arbre a choisi tout seul, parmi quatorze variables, de couper
d'abord sur celle-là — exactement ce que le tableau de la séance 4.2 montrait.

C'est une suite de **règles métier**, transposable telle quelle dans une note
de service.

In [ ]:
plt.figure(figsize=(7, 4))
dessin = plot_tree(arbre, feature_names=list(X.columns), class_names=["reste", "part"],
                   max_depth=2,          ## 2 niveaux : lisible a l'ecran
                   filled=True, impurity=False, fontsize=7)

# On efface la ligne "value" : le detail des effectifs par classe n'apporte
# rien tant qu'on lit l'arbre comme une suite de questions
for boite in dessin:
    boite.set_text("\n".join(ligne for ligne in boite.get_text().split("\n")
                              if not ligne.startswith("value")))
plt.show()

Chaque boîte contient trois choses, et seulement trois : la **question**, le
nombre d'abonnés qui arrivent là (`samples`), et la **réponse** de l'arbre pour
ce groupe (`class`). Les branches de gauche répondent « oui » à la question,
celles de droite « non ».

## 2. Choisir la profondeur sans tricher

Pourquoi `max_depth=3` ? Pour aucune bonne raison : nous l'avons décidé.

Tentation : essayer toutes les profondeurs sur le test et garder la meilleure.
**Impossible** — le test aurait servi à décider, il ne mesurerait plus rien.

La **validation croisée** résout ça sans y toucher. On découpe le jeu
d'apprentissage en cinq parts égales, puis on répète cinq fois : entraîner sur
quatre parts, mesurer sur la cinquième, changer de part. Chaque abonné
d'apprentissage sert donc **une fois à mesurer et quatre fois à entraîner**, et
on retient la moyenne des cinq mesures.

In [ ]:
for profondeur in [2, 3, 4, 5, 8, None]:   ## None = aucune limite
    scores = cross_val_score(
        DecisionTreeClassifier(max_depth=profondeur, random_state=42),
        X_tr, y_tr,          ## sur l'APPRENTISSAGE : le test reste ferme
        cv=5,                ## cinq decoupages successifs
        scoring="f1")        ## le F1 de la seance 4.2
    print(f"profondeur {str(profondeur):<5} F1 moyen {scores.mean():.3f}")

### Ce que ce tableau dit, et ce qu'il ne dit pas

Deux enseignements sont solides :

- **Sans aucune limite, l'arbre est le plus mauvais** (F1 0,509) alors qu'il
  atteint 0,993 sur ses propres données d'apprentissage. C'est le
  surapprentissage de la séance 4.1, sur un problème de décision cette fois.
- **La profondeur 3, celle que nous venions de lire, est le plus mauvais des
  réglages limités** (0,479). Nous l'avions choisie au hasard, et le hasard a
  mal choisi.

Un troisième point demande plus de prudence : entre les profondeurs 2 (0,586),
4 (0,558) et 5 (0,566), les écarts sont de deux ou trois centièmes. La
**question 13 les comparera à l'écart entre les cinq découpages** — et montrera
qu'ils sont du même ordre. Autrement dit, la validation croisée ne les
départage pas.

Quand une mesure ne tranche pas, on tranche sur autre chose. Ici : un arbre de
profondeur 2 ne pose que deux questions, un peu court pour une note de
service. **Nous retenons 4**, en sachant que 2 aurait été défendable.

In [ ]:
bon = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr, y_tr)
libre = DecisionTreeClassifier(random_state=42).fit(X_tr, y_tr)   ## aucune limite

for nom, m in {"profondeur 4": bon, "sans limite": libre}.items():
    print(f"{nom:<14} justesse {accuracy_score(y_te, m.predict(X_te)):.3f}  "
          f"F1 {f1_score(y_te, m.predict(X_te)):.3f}")

Le test, lui, n'avait pas été ouvert pour choisir. Il peut donc servir à ce
pour quoi il est fait : **annoncer un chiffre**. F1 de 0,523 pour l'arbre
bridé, 0,501 pour l'arbre libre.

## 3. Qu'est-ce qui fait la prédiction ?

> ⚠️ Deux noms se ressemblent et n'ont rien à voir : **`mensuel`** est le
> *montant de la facture mensuelle*, **`contrat_mensuel`** dit que l'abonné
> n'est *engagé sur aucune durée*. Ils vont tous les deux apparaître.

### Première réponse : l'importance native

Un arbre sait dire, gratuitement, combien chacune de ses coupures l'a fait
progresser. Demandons-le à l'arbre **sans limite**, celui qui a coupé partout.

In [ ]:
# feature_importances_ : le gain cumule de toutes les coupures sur la variable
native = pd.Series(libre.feature_importances_, index=X.columns)

native.sort_values(ascending=False).head(5).round(3)

La facture mensuelle arrive en tête (0,258), suivie de la facture cumulée
(0,237). Le contrat n'est que troisième. Conclusion apparente : **c'est le
montant qui fait partir les clients**.

Gardez cette phrase en tête trente secondes.

### Deuxième réponse : l'importance par permutation

Principe : on mélange une colonne au hasard — ce qui détruit son information
sans rien changer d'autre — et on regarde **combien le modèle perd**, sur le
jeu de test. Une variable utile fait chuter la performance ; une variable
inutile ne change rien.

In [ ]:
# On melange une colonne et on regarde ce que le modele PERD, en F1
perm = permutation_importance(libre, X_te, y_te,   ## sur le TEST
                              n_repeats=5, random_state=42, scoring="f1")

pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(5).round(4)

**`contrat_mensuel` passe premier**, loin devant (0,148). Les deux factures
reculent.

### Pourquoi les deux classements diffèrent

`feature_importances_` additionne le gain de **toutes les coupures** où la
variable apparaît. Or une variable continue à des milliers de valeurs
distinctes offre des milliers de coupures possibles, et l'arbre s'en ressert à
chaque niveau ; une colonne 0/1 n'offre **qu'une seule** coupure, utilisable
une fois. Le total favorise donc mécaniquement les variables continues,
indépendamment de leur utilité réelle.

La démonstration tient en une ligne, et c'est la question 9 : ajoutez à `X` une
colonne de nombres tirés **au hasard**, sans aucun lien avec le départ. Sur
l'arbre sans limite, elle arrive **première** en importance native.

La permutation mesure autre chose : ce que le modèle **perd** quand on lui
retire l'information, sur des données qu'il n'a jamais vues.

> ⚠️ **Croyez la permutation.** Et remarquez qu'elle confirme le tableau de la
> séance 4.2 : 42,7 % de départs au contrat mensuel contre 2,8 % à deux ans.
> Deux chemins différents, une seule réponse.

### Et surtout : la permutation marche sur n'importe quel modèle

`feature_importances_` n'existe **que** pour les modèles construits à base
d'arbres. Une régression linéaire ou logistique n'a pas cet attribut du tout.

La permutation, elle, ne demande rien au modèle : juste de savoir prédire. On
mélange une colonne, on redemande une prédiction, on mesure la perte. Ça
fonctionne donc **à l'identique** sur la régression logistique de la séance
4.2 — et sur une régression linéaire, et sur tout ce que vous croiserez
ensuite.

In [ ]:
# Le modele de la seance 4.2, remonte a l'identique
logistique = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logistique.fit(X_tr, y_tr)

pl = permutation_importance(logistique, X_te, y_te,   ## meme fonction, autre modele
                            n_repeats=5, random_state=42, scoring="f1")
pd.Series(pl.importances_mean, index=X.columns).sort_values(ascending=False).head(5).round(4)

**Les mêmes trois variables en tête** — `anc`, `contrat_mensuel`,
`internet_fibre` — obtenues par une famille de modèles qui n'a rien à voir avec
un arbre.

C'est ce qui donne à une recommandation sa solidité : deux méthodes
indépendantes, un même verdict. À l'inverse, une conclusion qui ne tient que
sur un modèle est une conclusion sur ce modèle, pas sur l'entreprise.

## 4. Une importance ne donne pas un sens

« Le contrat compte » ne se décide pas. Il faut savoir **dans quel sens** et
**de combien**. C'est ce que montre une **dépendance partielle** : on fait
varier une seule variable, tout le reste figé, et on regarde la probabilité
que le modèle prédit.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

# On fait varier anc seule, tout le reste fige : le SENS de l'effet
PartialDependenceDisplay.from_estimator(bon, X_te, ["anc"], ax=ax)
plt.title("Risque de depart predit, selon l'anciennete")
plt.show()

La courbe descend par paliers — c'est un arbre, elle ne peut faire que ça — et
l'essentiel de la descente se joue **sur les premiers mois**.

Vérifions sur les taux réels, ce qui est toujours plus convaincant qu'une
courbe de modèle :

In [ ]:
tranches = pd.cut(tel["anc"], [0, 12, 24, 48, 72])   ## comme au bloc 2

(tel.groupby(tranches, observed=True)["churn"].mean() * 100).round(1)

**47,7 % la première année, 28,7 % la deuxième, 20,4 % entre deux et quatre
ans, 9,5 % au-delà.** Le risque ne disparaît jamais complètement, mais il est
divisé par cinq entre la première et la cinquième année.

Voilà une phrase de note de direction : « l'effort de rétention doit porter en
priorité sur les douze premiers mois, où se concentre près de la moitié des
départs. »

## 5. L'erreur qui ne prévient pas

In [ ]:
sur_appr = permutation_importance(libre, X_tr, y_tr,   ## le mauvais jeu !
                                  n_repeats=3, random_state=42, scoring="f1")

print("mesuree sur l'apprentissage :")
print(pd.Series(sur_appr.importances_mean, index=X.columns).nlargest(4).round(4))

`mensuel` et `total` reviennent en tête, `contrat_mensuel` retombe quatrième :
**le classement de la section 3 est défait.**

Mesurée sur l'apprentissage, l'importance récompense ce que le modèle a
**mémorisé**, pas ce qui l'aide à généraliser — et l'arbre sans limite a tout
mémorisé. Aucun message d'erreur ne signale quoi que ce soit.

> ⚠️ **Une importance se mesure toujours sur des données que le modèle n'a
> jamais vues.** C'est la règle de la séance 4.1, appliquée à
> l'interprétation.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| un arbre lisible | `DecisionTreeClassifier(max_depth=3)` |
| lire ses règles | `print(export_text(a, feature_names=list(X.columns)))` |
| le dessiner | `plot_tree(a, feature_names=..., filled=True, impurity=False)` |
| choisir un réglage sans toucher au test | `cross_val_score(modele, X_tr, y_tr, cv=5, scoring="f1")` |
| l'importance native (biaisée, arbres seulement) | `m.feature_importances_` |
| l'importance honnête, sur **tout** modèle | `permutation_importance(m, X_te, y_te, n_repeats=5, scoring="f1")` |
| le sens de l'effet | `PartialDependenceDisplay.from_estimator(m, X_te, ["anc"])` |

## Les deux mesures d'importance

| Mesure | Ce qu'elle vaut | Ses limites |
|---|---|---|
| `feature_importances_` | gratuite, immédiate | n'existe **que** pour les arbres ; calculée sur l'apprentissage ; **favorise les variables à nombreuses valeurs** — ici `mensuel` et `total` |
| permutation | mesurée sur le test, sur la métrique qui vous intéresse, et sur **n'importe quel** modèle | plus lente ; sensible à la métrique choisie |

## Les trois phrases à retenir

1. **Un arbre se lit.** C'est sa vraie force : `export_text` produit des règles
   transposables telles quelles dans une note de service.

2. **L'importance native se laisse berner.** Sur l'arbre sans limite, elle
   classe `mensuel` et `total` en tête ; la permutation met `contrat_mensuel`
   premier — ce que le tableau de la séance 4.2 disait déjà.

3. **Une importance ne donne pas un sens.** « Le contrat compte » ne se décide
   pas ; « le risque de départ se concentre sur la première année » se décide.